In [ ]:
#如何优化你的pandas code，让其运行速度更快
"""
pandas 的全称为 python data analysis library,是类似于R和SAS语言的python 数据分析工具

pandas 主要进行数据分析的结构有两种，一种是DataFrame，另一种是Series

DataFrame：是一个二维数组标记轴，是Series的容器（或者说，DataFrame是有行有列的矩阵，列有列名标签，行有索引标签

Series：在pandas中，一个DataFrame的一行或者一列就是一个pandas series，一个带有轴标签的一堆数组

从慢到快，能对pandas的DataFrame结构进行优化的方法有：

1． 在用索引的DataFrame行上的Crude looping 
2． 用iterrows()循环 
3． 用 apply()循环 
4． Pandas Series矢量化 
5． NumPy数组矢量化

1.crude loop（几乎没有任何内置优化空间）
在对你的pandas代码进行分析时，有必要引入一个jupyternote专用的%timeit（jupyternote中的所有魔法命令都以%开始，
%作用于一行，%%作用于整个cell

2.使用itterrows() 进行循环

如果当循环是必须的时候，用itterrows() 可以更好的遍历行，itterrows() 是一个生成器，遍历DataFrame的所有行，并且返回

索引，除了包含行自身的对象，itterrows() 是用pandas DataFrame进行内置优化，但却是大多数标准函数最不高效的方式，但是

像对于crude looping（手动遍历）要方便很多很多

3.使用.apply方法

.apply 是一个比itterrows（）更好的一个方法，应用一个函数沿着DataFrame的某个特定轴线（行或列），.apply 虽然也是固有

的通过行进行循环，但是却比itterrows() 的内部优化要好不少，一般来说，使用.apply能比使用.itterrows() 减少将近

一半的时间，这里又引入一个特定的魔法工具 %iprun,可以具体显示运行的时间

4.pandas series 矢量化：

目标：减少函数迭代的数量，Pandas的基本单位是DataFrame和Series，二者都基于数组

基本单元的固有结构转换成内置的设计对整个数组进行操作的pandas函数，而不是按照各个值的顺序（简称标量）。矢量化是对

正个数组进行操作的过程。

pandas包括一个总体的矢量化函数集合，Pandas包含一个总体的矢量化函数集合，从数学运算到聚合和字符串函数
（可用函数的扩展列表，查看Pandas docs）。对Pandas Series和DataFrame的操作进行内置优化。
结果，使用矢量Pandas函数几乎总是会用自定义的循环实现类似的功能。

5.使用numpy 进行数组矢量化

pandas矢量库可以完整日常绝大部分数据处理优化工作，但是如果将处理速度列为最高优先级的话，

Pandas series矢量化可以完成日常计算优化的绝大多数需要。然而，如果速度是最高优先级，那么可以以NumPy Python库的形式调用援军。

NumPy库，将自己描述为一个“Python科学计算的基本包”，在后台执行优化操作，预编译C语言代码。跟Pandas一样，NumPy操作数组对象（简称ndarrays）
；然而，它省去了Pandas series操作所带来的大量资源开销，如索引、数据类型检查等。因此，NumPy数组的操作可以明显快于pandas series的操作。

当Pandas series提供的额外功能不是很关键的时候，NumPy数组可以用于替代Pandas series。例如，
Haversine函数矢量化实现不使用索引的经度和纬度系列，因此没有那些索引，也不会导致函数中断。
通过比较，我们所做的操作如DataFrame的连接，它需要按索引来引用值，可能需要坚持使用Pandas对象。

仅仅是使用Pandas series 的values的方法，把纬度和经度数组从Pandas series转换到NumPy数组。
就像series矢量化一样，通过NumPy数组直接进入函数将可以让Pandas对整个矢量应用函数。



这给我们带来了一些关于优化Pandas代码的基本结论：

1.避免循环；它们很慢，而且在大多数情况下是不必要的。
2.如果必须使用循环，用 apply(),而不是迭代函数。
3.矢量化通常优于标量运算。在Pandas中的大部分常见操作都可以矢量化。
4.NumPy数组矢量化操作比Pandas series更有效。
当然，以上并不是Pandas所有可能优化的全面清单。
更爱冒险的用户或许可以考虑进一步用Cython改写函数，或者尝试优化函数的各个组件。然而，这些话题超出了这篇文章的范围。

关键的是，在开始一次宏大的优化冒险之前，要确保正在优化的函数实际上是你希望在长期运行中使用的函数。
引用XKCD不朽的名言：“过早优化是万恶之源”。"""




In [ ]:
import pandas as pd
import numpy as np
import os

class DataProcessor:
    def __init__(self,data_dir = './data/'):
        self.data_dir = data_dir
        self.raw_data = None
        self.prices = None

    def load_raw(self):
        df_list = []

        for file in os.listdir(self.data_dir):
            if file.endswith('.csv'):
                file_path = os.path.join(self.data_dir,file)
                df = pd.read_csv(file_path,skiprows = [1])

                ticker = file.replace('.csv','')
                #ticker = str(file.replace(".csv", ""))
                df['ticker'] = ticker
                df['date'] = pd.to_datetime(df['Date'])
                df_list.append(df)

        self.raw_data = pd.concat(df_list,ignore_index = True)
        return self.raw_data
        # why here choose to return a DataFrame instead of a list?
        #what is the difference between this and the original code in day2


    def process_prices(self,price_col = 'Close',fill_method = 'ffill'):
        if self.raw_data is None:
            self.load_raw()
        prices = {}
        for ticker, group in self.raw_data.groupby('ticker'):
            #self.raw_data.groupby('ticker').apply(lambda x : prices[x.iloc(0)] = self.raw_data.set_index('date')[price_col])
            #这里 ticker 返还的首先是ticker，group返还的是这个ticker所对应的所有的数据
            prices[ticker] = group.set_index('date')[price_col]

        self.prices = pd.DataFrame(prices).sort_index()

        if fill_method == 'ffill':
            self.prices = self.prices.ffill()

        return self.prices

    def process_returns(self):
        """输出日收益率矩阵"""
        if self.prices is None:
            self.process_prices()

        return self.prices.pct_change().dropna()
        #what is the meaning of this sentence
        #pct_change() is used for caculagte the difference of percentage
        #dropna() means to drop the first day's data as the zero day does not exist

    def get_stats(self):
        if self.prices is None:
            self.process_prices()
        returns =  self.process_returns()

        stats = pd.DataFrame({
            'mean':returns.mean(),
            'var':returns.var(),
            'skew':returns.skew()
        })

        return stats
if __name__ == '__main__':
    
    processor = DataProcessor("./data/")
    
    # 测试所有功能
    raw = processor.load_raw()
    print("1. 原始数据加载完成:", raw.shape)
    
    prices = processor.process_prices()
    print("\n2. 价格矩阵:")
    print(prices.head())
    
    returns = processor.process_returns()
    print("\n3. 收益率矩阵:")
    print(returns.head())    
    stats = processor.get_stats()
    print("\n4. 统计指标:")
    print(stats)
        
        
                
    

In [ ]:
def process_prices(self, price_col='Close', fill_method='ffill'):
    if self.raw_data is None:
        self.load_raw()
    
    # 🔥 优化核心：groupby('ticker') + apply，直接生成目标DataFrame
    self.prices = self.raw_data.groupby('ticker').apply(
        lambda x: x.set_index('date')[price_col]  # 和原for循环逻辑完全一样
    ).T.sort_index()  # .T 转置，让日期变行，股票变列（和原版一致）

    if fill_method == 'ffill':
        self.prices = self.prices.ffill()
    
    return self.prices



In [ ]:
class DataProcessor:
    def __init__(self, data_dir='./data/'):
        self.data_dir = data_dir
    
    def load_raw(self):
        """基于已知数据结构的清洗（简单直接）"""
        df_list = []
        
        for file in os.listdir(self.data_dir):
            if file.endswith('.csv'):
                # 已知：utf-8 编码，有垃圾行，日期列叫 Date
                df = pd.read_csv(
                    os.path.join(self.data_dir, file), 
                    skiprows=[1]
                )
                df['ticker'] = file.replace('.csv', '')
                df['date'] = pd.to_datetime(df['Date'])
                df_list.append(df)
        
        return pd.concat(df_list, ignore_index=True)
    
    # ... 其他方法保持不变

In [ ]:
#在现实中，除非是一个非常固定和规范的流程，否则不要总是想着自动化流程，
#在你还不清楚你具体需要分析的数据类型的时候，使用jupyterlab的df.head(), df.info()l来认识其具体的数据结构
